# Práctica 5: Fine-tuning de ALBERT para análisis de sentimientos

En esta práctica se realiza fine-tuning de un modelo transformer preentrenado para clasificar oraciones como positivas o negativas.

## 1. Imports

In [33]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding
from transformers import Trainer, TrainingArguments

## 2. Configuración general

In [34]:
MODEL_CHECKPOINT = "albert/albert-base-v2"
DATASET_NAME = "nyu-mll/glue"
DATASET_CONFIG = "sst2"

OUTPUT_DIR = "sentiment-trainer"
FINAL_MODEL_DIR = "modelo-sentimientos"

SEED = 42
TRAIN_SIZE = 3000
EVAL_SIZE = 500

## 3. Carga del dataset

In [35]:
raw_datasets = load_dataset(DATASET_NAME, DATASET_CONFIG)

In [36]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [37]:
raw_datasets["train"][0]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0}

In [38]:
raw_datasets["train"][9105]

{'sentence': 'has turned out nearly 21/2 hours of unfocused , excruciatingly tedious cinema that , half an hour in , starts making water torture seem appealing ',
 'label': 0,
 'idx': 9105}

## 4. Tokenizador

In [39]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

In [40]:
tokenizer("This cheese is good.")

{'input_ids': [2, 48, 7372, 25, 254, 9, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

## 5. Tokenización del dataset

In [41]:
def tokenize_function(example):
    return tokenizer(example["sentence"], truncation=True)

In [42]:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [43]:
tokenized_datasets["train"][0]

{'sentence': 'hide new secretions from the parental units ',
 'label': 0,
 'idx': 0,
 'input_ids': [2, 3077, 78, 27467, 18, 37, 14, 21207, 1398, 3],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## 6. Padding dinámico

In [44]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 7. Carga del modelo

In [45]:
id2label = {
    0: "negativo",
    1: "positivo",
}

label2id = {
    "negativo": 0,
    "positivo": 1,
}

In [46]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert/albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.bias             | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 8. Métrica de evaluación
Definimos manualmente la métrica de accuracy. Esta métrica mide la proporción de ejemplos clasificados correctamente. Se implementó directamente con NumPy para evitar depender de la librería `evaluate`, que causó conflictos con `torchvision` en el entorno local.

In [47]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()

    return {
        "accuracy": accuracy,
    }

## 9. Subconjuntos de entrenamiento y validación

In [48]:
small_train_dataset = (
    tokenized_datasets["train"]
    .shuffle(seed=SEED)
    .select(range(TRAIN_SIZE))
)

small_eval_dataset = (
    tokenized_datasets["validation"]
    .shuffle(seed=SEED)
    .select(range(EVAL_SIZE))
)

In [49]:
small_train_dataset[0]

{'sentence': 'klein , charming in comedies like american pie and dead-on in election , ',
 'label': 1,
 'idx': 32326,
 'input_ids': [2,
  10201,
  13,
  15,
  13275,
  19,
  25271,
  101,
  189,
  5470,
  17,
  828,
  8,
  218,
  19,
  776,
  13,
  15,
  3],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [50]:
small_eval_dataset[0]

{'sentence': 'it gets onto the screen just about as much of the novella as one could reasonably expect , and is engrossing and moving in its own right . ',
 'label': 1,
 'idx': 726,
 'input_ids': [2,
  32,
  3049,
  1204,
  14,
  2324,
  114,
  88,
  28,
  212,
  16,
  14,
  21345,
  28,
  53,
  110,
  19531,
  4186,
  13,
  15,
  17,
  25,
  1957,
  18801,
  17,
  1219,
  19,
  82,
  258,
  193,
  13,
  9,
  3],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

## 10. Configuración del entrenamiento

In [51]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    report_to="none",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    dataloader_pin_memory=False,
)

## 11. Entrenamiento

In [52]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [53]:
train_results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.522614,0.758000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [54]:
train_results

TrainOutput(global_step=375, training_loss=0.5555997721354167, metrics={'train_runtime': 546.4961, 'train_samples_per_second': 5.49, 'train_steps_per_second': 0.686, 'total_flos': 4505165589600.0, 'train_loss': 0.5555997721354167, 'epoch': 1.0})

## 12. Evaluación

In [55]:
from transformers.utils.notebook import NotebookProgressCallback

trainer.remove_callback(NotebookProgressCallback)
eval_results = trainer.evaluate()

In [56]:
eval_results

{'eval_loss': 0.5226144790649414,
 'eval_accuracy': 0.758,
 'eval_runtime': 38.0039,
 'eval_samples_per_second': 13.157,
 'eval_steps_per_second': 1.658,
 'epoch': 1.0}

## 13. Guardado del modelo

In [57]:
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('modelo-sentimientos/tokenizer_config.json',
 'modelo-sentimientos/tokenizer.json')

## 14. Predicción manual

In [58]:
def predict_sentiment(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]

    negative_score = probs[0].item()
    positive_score = probs[1].item()

    if positive_score > negative_score:
        predicted_label = "positivo"
        confidence = positive_score
    else:
        predicted_label = "negativo"
        confidence = negative_score

    return {
        "texto": text,
        "predicción": predicted_label,
        "confianza": confidence,
        "score_negativo": negative_score,
        "score_positivo": positive_score,
    }

In [59]:
predict_sentiment("This movie was really good.")

{'texto': 'This movie was really good.',
 'predicción': 'positivo',
 'confianza': 0.944129467010498,
 'score_negativo': 0.05587056279182434,
 'score_positivo': 0.944129467010498}

In [60]:
predict_sentiment("This class was boring and terrible.")

{'texto': 'This class was boring and terrible.',
 'predicción': 'negativo',
 'confianza': 0.8620553612709045,
 'score_negativo': 0.8620553612709045,
 'score_positivo': 0.13794468343257904}

In [61]:
predict_sentiment("The food was okay, but the service was awful.")

{'texto': 'The food was okay, but the service was awful.',
 'predicción': 'negativo',
 'confianza': 0.8602356910705566,
 'score_negativo': 0.8602356910705566,
 'score_positivo': 0.13976427912712097}